# Manformer — Data Download

Downloads three co-registered datasets for a small subset of Florida mangrove tiles:

| Output | Source | Resolution | Purpose |
|--------|--------|------------|---------|
| `data/naip/tile_XXX_naip.tif` | USDA NAIP DOQQ | **1 m**, 4-band RGBN | HR image input to Paraformer |
| `data/labels/tile_XXX_label.tif` | ESA WorldCover v200 | **1 m** (NN-resampled from 10 m) | LR label input to Paraformer |
| `data/sentinel2_rgb/tile_XXX_s2.tif` | Sentinel-2 SR | **10 m**, RGB | Visual reference only |

The tile footprint is `TILE_SIZE_M × TILE_SIZE_M` metres (default 4096 m), kept small for a fast
first test. Tiles are filtered so they contain at least one ESA Mangrove (class 95) pixel.

Two CSVs are written at the end:
- `dataset/florida_manformer_train.csv`
- `dataset/florida_manformer_val.csv`

## 0 — Imports

In [1]:
import ee
import os
import random
import pandas as pd
import geemap

# Ensure paths resolve relative to this notebook regardless of where Jupyter was launched
os.chdir(os.path.dirname(os.path.abspath('data_download.ipynb')))

## 1 — GEE Authentication & Initialisation

In [2]:
# ee.Authenticate()  # Uncomment once if token is not already saved
ee.Initialize(project='e4e-mangrove')
print('GEE initialised.')

GEE initialised.


## 2 — Configuration & Manual Tile Coordinates

Paste the center points of the areas you picked in the GEE browser.
Each entry is `(latitude, longitude)`. The notebook builds a
`TILE_SIZE_M × TILE_SIZE_M` bounding box around each point and downloads it.

In [3]:
# ── Tile size ─────────────────────────────────────────────────────────────────
TILE_SIZE_M = 4096   # metres; 4096 m → 4096×4096 px at 1 m (~40 MB NAIP each)

# ── Manually selected tile centers (lat, lon) ─────────────────────────────────
# Picked from GEE browser. GEE returns [lon, lat] — stored here as (lat, lon).
TILE_CENTERS = [
    (26.507492644732974, -81.9223766782635),   # 1 — Fort Myers area
    (26.90510405971088,  -82.05732250848348),   # 2 — Charlotte Harbor / Port Charlotte
    (27.57319193976069,  -82.53825151333035),   # 3 — Tampa Bay / Sarasota coast
    (26.0923321441167,   -81.73339008631058),   # 4 — Naples / Marco Island
    (25.583277426837988, -80.36537701427653),   # 5 — Everglades / Biscayne coast
]

# ── Output directories ────────────────────────────────────────────────────────
BASE_DIR  = os.getcwd()
DATA_DIR  = os.path.join(BASE_DIR, 'data')
NAIP_DIR  = os.path.join(DATA_DIR, 'naip')
LABEL_DIR = os.path.join(DATA_DIR, 'labels')
S2_DIR    = os.path.join(DATA_DIR, 'sentinel2_rgb')
CSV_DIR   = os.path.join(BASE_DIR, 'dataset')

for d in [NAIP_DIR, LABEL_DIR, S2_DIR, CSV_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Tile size        : {TILE_SIZE_M} m  →  {TILE_SIZE_M}×{TILE_SIZE_M} px at 1 m')
print(f'Tiles to download: {len(TILE_CENTERS)}')
for i, (lat, lon) in enumerate(TILE_CENTERS):
    print(f'  [{i+1}] lat={lat:.5f}, lon={lon:.5f}')

Tile size        : 4096 m  →  4096×4096 px at 1 m
Tiles to download: 5
  [1] lat=26.50749, lon=-81.92238
  [2] lat=26.90510, lon=-82.05732
  [3] lat=27.57319, lon=-82.53825
  [4] lat=26.09233, lon=-81.73339
  [5] lat=25.58328, lon=-80.36538


## 3 — Build Tile Geometries from Center Points

In [4]:
def make_tile_geom(lat, lon, size_m):
    """Build a size_m × size_m bounding box centred on (lat, lon) in EPSG:3857."""
    half = size_m / 2
    center = ee.Geometry.Point([lon, lat])
    # Project to metres, buffer by half-size, then take the bounding box
    geom = center.buffer(half * (2 ** 0.5), 1).bounds()
    return geom

coords_array = [(lat, lon) for lat, lon in TILE_CENTERS]
print(f'Ready to download {len(coords_array)} tiles.')

Ready to download 5 tiles.


## 4 — Build Image Collections

In [5]:
# Bounding box derived from the tile centers — used to narrow collection queries
lats = [lat for lat, lon in TILE_CENTERS]
lons = [lon for lat, lon in TILE_CENTERS]
region_bounds = ee.Geometry.BBox(min(lons) - 0.5, min(lats) - 0.5,
                                  max(lons) + 0.5, max(lats) + 0.5)

# ── NAIP (HR input, ~1 m, RGBN) ──────────────────────────────────────────────
naip = (
    ee.ImageCollection('USDA/NAIP/DOQQ')
    .filterBounds(region_bounds)
    .filterDate('2020-01-01', '2023-01-01')
    .filter(ee.Filter.listContains('system:band_names', 'N'))
    .mosaic()
    .select(['R', 'G', 'B', 'N'])
)

# ── ESA WorldCover label (10 m native; downloaded at 1 m via NN resampling) ──
esa_label = (
    ee.ImageCollection('ESA/WorldCover/v200')
    .first()
    .select('Map')
    .byte()
)

# ── Sentinel-2 RGB (reference only, 10 m) ────────────────────────────────────
s2_rgb = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(region_bounds)
    .filterDate('2023-01-01', '2024-01-01')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .select(['B4', 'B3', 'B2'])
    .rename(['Red', 'Green', 'Blue'])
)

print('Image collections ready.')
print('NAIP bands  :', naip.bandNames().getInfo())
print('ESA bands   :', esa_label.bandNames().getInfo())
print('S2 RGB bands:', s2_rgb.bandNames().getInfo())

Image collections ready.
NAIP bands  : ['R', 'G', 'B', 'N']
ESA bands   : ['Map']
S2 RGB bands: ['Red', 'Green', 'Blue']


## 5 — Download Loop

For each tile:
1. **NAIP** at `scale=1` (native 1 m) → HR image for Paraformer
2. **ESA label** at `scale=1` → GEE nearest-neighbour resamples the 10 m label to 1 m,
   producing the same pixel dimensions as NAIP (required by `StreamingGeospatialDataset`)
3. **S2 RGB** at `scale=10` → reference only, not used in training

Files that already exist are skipped, so the cell is safe to re-run after interruptions.

In [6]:
naip_fns  = []
label_fns = []

for i, (lat, lon) in enumerate(coords_array):
    geom     = make_tile_geom(lat, lon, TILE_SIZE_M)
    tile_id  = f'{i + 1:03d}'
    naip_fn  = os.path.join(NAIP_DIR,  f'tile_{tile_id}_naip.tif')
    label_fn = os.path.join(LABEL_DIR, f'tile_{tile_id}_label.tif')
    s2_fn    = os.path.join(S2_DIR,    f'tile_{tile_id}_s2.tif')

    print(f'\n[{i + 1}/{len(coords_array)}] Tile {tile_id}  ({lat:.4f}, {lon:.4f})')

    # 1. NAIP — HR image (1 m, 4-band RGBN)
    if os.path.exists(naip_fn):
        print(f'  NAIP already exists, skipping.')
    else:
        print(f'  Downloading NAIP (1 m, ~{TILE_SIZE_M}×{TILE_SIZE_M} px)…')
        try:
            geemap.download_ee_image(
                image=naip, filename=naip_fn,
                region=geom, crs='EPSG:3857', scale=1
            )
            print(f'  Saved: {naip_fn}')
        except Exception as e:
            print(f'  NAIP download failed: {e}  — skipping tile.')
            continue

    # 2. ESA label — resampled to 1 m (same pixel dims as NAIP)
    if os.path.exists(label_fn):
        print(f'  Label already exists, skipping.')
    else:
        print(f'  Downloading ESA label (NN-resampled to 1 m)…')
        try:
            geemap.download_ee_image(
                image=esa_label, filename=label_fn,
                region=geom, crs='EPSG:3857', scale=1
            )
            print(f'  Saved: {label_fn}')
        except Exception as e:
            print(f'  Label download failed: {e}  — skipping tile.')
            continue

    # 3. S2 RGB — reference only (10 m)
    if os.path.exists(s2_fn):
        print(f'  S2 RGB already exists, skipping.')
    else:
        print(f'  Downloading Sentinel-2 RGB (10 m, reference)…')
        try:
            geemap.download_ee_image(
                image=s2_rgb, filename=s2_fn,
                region=geom, crs='EPSG:3857', scale=10
            )
            print(f'  Saved: {s2_fn}')
        except Exception as e:
            print(f'  S2 RGB download failed (non-critical): {e}')

    naip_fns.append(naip_fn)
    label_fns.append(label_fn)

print(f'\nComplete tile pairs: {len(naip_fns)} / {len(coords_array)}')


[1/5] Tile 001  (26.5075, -81.9224)


c:\Users\adytc\anaconda3\envs\mangrove\Lib\site-packages\geemap\common.py:12434: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/104 tiles [00:00<?]

c:\Users\adytc\anaconda3\envs\mangrove\Lib\site-packages\geedim\image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\data\naip\tile_001_naip.tif


...

  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\data\naip\tile_005_naip.tif


2021:   0%|          |0/26 tiles [00:00<?]

  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\data\labels\tile_005_label.tif


  0%|          |0/12 tiles [00:00<?]

  Saved: c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\data\sentinel2_rgb\tile_005_s2.tif

Complete tile pairs: 5 / 5


## 6 — Generate Train / Val CSVs

80 / 20 split (at least 1 val tile). CSVs use columns `image_fn, label_fn` — the format
expected by `StreamingGeospatialDataset` in `trainer.py`.

In [7]:
random.seed(42)
pairs = list(zip(naip_fns, label_fns))
random.shuffle(pairs)

n_val       = max(1, len(pairs) // 5)   # 20 %
val_pairs   = pairs[:n_val]
train_pairs = pairs[n_val:]

train_csv = os.path.join(CSV_DIR, 'florida_manformer_train.csv')
val_csv   = os.path.join(CSV_DIR, 'florida_manformer_val.csv')

pd.DataFrame(train_pairs, columns=['image_fn', 'label_fn']).to_csv(train_csv, index=False)
pd.DataFrame(val_pairs,   columns=['image_fn', 'label_fn']).to_csv(val_csv,   index=False)

print(f'Train CSV : {len(train_pairs)} tiles  →  {train_csv}')
print(f'Val CSV   : {len(val_pairs)} tile(s) →  {val_csv}')
print()
print('Next step: run train.py')
print('  python train.py \\')
print(f'    --train_list {train_csv} \\')
print(f'    --val_list   {val_csv} \\')
print('    --savepath   experiments/run_001 \\')
print('    --gpu        0')

Train CSV : 4 tiles  →  c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\dataset\florida_manformer_train.csv
Val CSV   : 1 tile(s) →  c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\dataset\florida_manformer_val.csv

Next step: run train.py
  python train.py \
    --train_list c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\dataset\florida_manformer_train.csv \
    --val_list   c:\vscode workspace\ml-mangrove\Scale Adaption\manformer\dataset\florida_manformer_val.csv \
    --savepath   experiments/run_001 \
    --gpu        0
